In [ ]:
import os, re, json, time, logging, warnings, random
from datetime import datetime
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import boto3
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
print(f"boto3   {boto3.__version__}")
print(f"pandas  {pd.__version__}")

In [ ]:
BASE_DIR    = Path("d:/Projeto-Doutorado-V2")
DATA_DIR    = BASE_DIR / "Data"
RESULTS_DIR = BASE_DIR / "Results"
LOGS_DIR    = BASE_DIR / "Logs"
FIGURES_DIR = RESULTS_DIR / "Figures" / "JACV_Construction"
TABLES_DIR  = RESULTS_DIR / "Tables"
AUDIT_DIR   = RESULTS_DIR / "Audit"

for d in [RESULTS_DIR, LOGS_DIR, FIGURES_DIR, TABLES_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLEAN_PATH      = DATA_DIR / "dataset_clean_cnj.json"
JACV_PATH       = DATA_DIR / "dataset_jacv.json"
CHECKPOINT_PATH = RESULTS_DIR / "jacv_generation_checkpoint.json"
RESULTS_JSON    = RESULTS_DIR / "jacv_construction_results.json"
LOG_PATH        = LOGS_DIR / "jacv_construction.log"
ENV_PATH        = BASE_DIR / ".env"
DATASET_CARD    = RESULTS_DIR / "jacv_dataset_card.json"
FLOW_TABLE_CSV  = TABLES_DIR / "jacv_flow_table.csv"
AUDIT_SAMPLE_CSV = AUDIT_DIR / "jacv_manual_audit_sample.csv"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(LOG_PATH, mode="a", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("JACV_Construction")
logger.info("Notebook 03_JACV_Dataset_Construction STARTED")

N_ANCHORS        = 200
CF_PER_ANCHOR    = 2
RANDOM_SEED      = 42
MAX_WORDS_FATO   = 600
MAX_WORDS_DIR    = 800
MAX_WORDS_PED    = 400
MAX_RETRIES      = 3
RETRY_DELAY      = 2.0
CHECKPOINT_FREQ  = 10
API_SLEEP        = 0.6
MAX_CF_REGEN     = 2
MIN_CF_CHAR_DELTA = 12
AUDIT_SAMPLE_SIZE = 60

GENERATOR_CHAIN = [
    ("Llama 3.3 70B",         "us.meta.llama3-3-70b-instruct-v1:0",             "meta"),
    ("Llama 4 Maverick 17B",      "us.meta.llama4-maverick-17b-instruct-v1:0",     "meta"),
    ("Mistral Large 3",           "us.mistral.mistral-large-3-675b-instruct",       "mistral"),
]

VALIDATOR_CHAIN = [
    ("Claude Sonnet 4",           "us.anthropic.claude-sonnet-4-20250514-v1:0",    "anthropic"),
    ("Claude 3.5 Haiku",      "us.anthropic.claude-3-5-haiku-20241022-v1:0",    "anthropic"),
    ("Mistral Large 3",       "us.mistral.mistral-large-3-675b-instruct",        "mistral"),
]

RESULTS = {
    "metadata": {
        "notebook": "03_JACV_Dataset_Construction",
        "pipeline_stage": 3,
        "started_at": datetime.now().isoformat(),
        "generator_chain": [m[0] for m in GENERATOR_CHAIN],
        "validator_chain": [m[0] for m in VALIDATOR_CHAIN],
        "n_anchors": N_ANCHORS,
        "cf_per_anchor": CF_PER_ANCHOR,
        "max_cf_regen": MAX_CF_REGEN,
        "audit_sample_size": AUDIT_SAMPLE_SIZE,
    }
}
logger.info(f"Config: {N_ANCHORS} anchors x {CF_PER_ANCHOR} CF")
print("Setup complete.")

## Section 1 — AWS Bedrock Client Setup & Connectivity Test

In [ ]:
def load_env(path):
    env = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                env[k.strip()] = v.strip()
    return env

env = load_env(ENV_PATH)
AWS_KEY    = env["AWS_ACCESS_KEY_ID"]
AWS_SECRET = env["AWS_SECRET_ACCESS_KEY"]
AWS_REGION = env["AWS_DEFAULT_REGION"]

bedrock = boto3.client(
    "bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
)
logger.info(f"AWS region: {AWS_REGION} | Key: {AWS_KEY[:8]}***")

def ping_model(model_name, model_id, family):
    try:
        if family == "anthropic":
            body = json.dumps({
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 8,
                "temperature": 0.0,
                "messages": [{"role": "user", "content": "Respond only READY"}],
            })
            resp = bedrock.invoke_model(modelId=model_id, body=body)
            txt = json.loads(resp["body"].read())["content"][0]["text"].strip()

        elif family == "meta":
            prompt = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\nRespond only READY\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
            body = json.dumps({
                "prompt": prompt,
                "max_gen_len": 8,
                "temperature": 0.0
            })
            resp = bedrock.invoke_model(modelId=model_id, body=body)
            txt = json.loads(resp["body"].read()).get("generation", "").strip()

        elif family == "mistral":
            body = json.dumps({
                "messages": [{"role": "user", "content": "Respond only READY"}],
                "max_tokens": 8,
                "temperature": 0.0,
            })
            resp = bedrock.invoke_model(modelId=model_id, body=body)
            txt = json.loads(resp["body"].read())["choices"][0]["message"]["content"].strip()

        else:
            raise ValueError(f"Unsupported family: {family}")

        print(f"Primary generator ({model_name}): CONNECTED -> {txt!r}")
        logger.info(f"Bedrock connectivity OK for {model_name}: {txt!r}")
        return True

    except Exception as e:
        print(f"WARNING: Primary generator ping failed ({e}). Fallback chain will be used.")
        logger.warning(f"Primary generator ping failed: {e}")
        return False

ping_model(*GENERATOR_CHAIN[0])

## Section 2 — Load Clean Data & Select Anchor Instances

In [ ]:
logger.info(f"Loading clean dataset: {CLEAN_PATH}")
with open(CLEAN_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df["is_recurso"] = df["is_recurso"].astype(bool)

for fld in ["fato", "direito", "pedido"]:
    df[f"{fld}_words"] = df[fld].astype(str).str.split().str.len()

logger.info(f"Loaded {len(df):,} records")

filter_conditions = (
    (df["fato_words"]    >= 80)  &
    (df["direito_words"] >= 150) &
    (df["pedido_words"]  >= 50)  &
    (df["direito_words"] <= 5000)
)

if "direito_is_outlier" in df.columns:
    filter_conditions &= (~df["direito_is_outlier"]) & (~df["fato_is_outlier"])

df_eligible = df[filter_conditions].copy()
logger.info(f"Eligible records: {len(df_eligible)}/{len(df)}")
print(f"Eligible: {len(df_eligible)}/{len(df)} | Non-Appeal: {(df_eligible.is_recurso==False).sum()} | Appeal: {(df_eligible.is_recurso==True).sum()}")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

n_each     = N_ANCHORS // 2
non_appeal = df_eligible[df_eligible.is_recurso==False].sample(min(n_each, (df_eligible.is_recurso==False).sum()), random_state=RANDOM_SEED)
appeal     = df_eligible[df_eligible.is_recurso==True].sample(min(n_each, (df_eligible.is_recurso==True).sum()), random_state=RANDOM_SEED)
anchors    = pd.concat([non_appeal, appeal]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

if "record_id" not in anchors.columns:
    anchors["record_id"] = ["REC_{:05d}".format(i) for i in anchors.index]

logger.info(f"Anchor set: {len(anchors)} ({(anchors.is_recurso==False).sum()} non-appeal + {(anchors.is_recurso==True).sum()} appeal)")
print(f"\nAnchor set: {len(anchors)} instances")
print(f"  fato_words    mean={anchors.fato_words.mean():.0f}")
print(f"  direito_words mean={anchors.direito_words.mean():.0f}")
print(f"  pedido_words  mean={anchors.pedido_words.mean():.0f}")

## Section 3 — JACV Perturbation Taxonomy & Prompt Templates

In [ ]:
PERTURBATION_TYPES = {
    "INCOHERENT": {
        "label": "INCOHERENT",
        "description": (
            "O pedido modificado NÃO decorre logicamente da fundamentação jurídica (direito). "
            "O provimento requerido não pode ser derivado dos argumentos e das normas legais citadas. "
            "Solicite um provimento incompatível com os dispositivos citados, atribua responsabilidade "
            "à parte errada ou pleiteie resultado não amparado pelos artigos invocados."
        ),
    },
    "CONTRADICTORY": {
        "label": "CONTRADICTORY",
        "description": (
            "O pedido modificado CONTRADIZ DIRETAMENTE o que a fundamentação jurídica sustenta. "
            "Inverta a conclusão lógica do direito mantendo o português jurídico natural."
        ),
    },
}

VALIDATION_LABELS = ["COHERENT", "INCOHERENT", "PARTIAL", "CONTRADICTORY"]

def truncate(text: str, max_words: int) -> str:
    words = str(text).split()
    if len(words) <= max_words:
        return str(text)
    truncated = " ".join(words[:max_words])
    last_sep = max(truncated.rfind(". "), truncated.rfind("; "))
    if last_sep > int(len(truncated) * 0.70):
        return truncated[:last_sep + 1].strip()
    return truncated.strip() + "..."

SYSTEM_GENERATION = (
    "Você é um especialista jurídico brasileiro. "
    "Escreva em português jurídico natural, formal e realista."
)

def build_generation_prompt(fato, direito, pedido, ptype):
    pdef = PERTURBATION_TYPES[ptype]
    label_map = {"INCOHERENT": "INCOERENTE", "CONTRADICTORY": "CONTRADITÓRIO"}
    label_ptbr = label_map[ptype]

    return "\n".join([
        "Você irá modificar o PEDIDO de uma petição judicial brasileira para criar uma perturbação controlada.",
        "",
        f"TAREFA: gere uma versão MODIFICADA do PEDIDO classificada como {label_ptbr} ({ptype}).",
        "",
        "REQUISITO DA PERTURBAÇÃO:",
        pdef["description"],
        "",
        "RESTRIÇÕES:",
        "- português jurídico brasileiro formal",
        "- manter fluência e plausibilidade",
        "- manter comprimento semelhante ao original (variação máxima de 30%)",
        "- não adicionar explicações, rótulos ou metadados",
        "- retornar APENAS o texto do pedido modificado",
        "",
        "---",
        "FATO:",
        truncate(fato, MAX_WORDS_FATO),
        "",
        "DIREITO:",
        truncate(direito, MAX_WORDS_DIR),
        "",
        "PEDIDO ORIGINAL (coerente):",
        truncate(pedido, MAX_WORDS_PED),
        "---",
        "",
        f"Gere agora o PEDIDO MODIFICADO ({ptype}):"
    ])

def build_validation_prompt(direito, pedido_cf):
    return "\n".join([
        "Avalie a coerência lógica entre DIREITO e PEDIDO.",
        "",
        "Classifique o PEDIDO com EXATAMENTE UMA das etiquetas:",
        "- COHERENT",
        "- INCOHERENT",
        "- PARTIAL",
        "- CONTRADICTORY",
        "",
        "DIREITO:",
        truncate(direito, MAX_WORDS_DIR),
        "",
        "PEDIDO:",
        truncate(pedido_cf, MAX_WORDS_PED),
        "",
        "Responda APENAS com UMA palavra: COHERENT, INCOHERENT, PARTIAL ou CONTRADICTORY."
    ])

def is_counterfactual_nontrivial(original_text, cf_text):
    original_text = str(original_text).strip()
    cf_text = str(cf_text).strip()
    if not cf_text:
        return False
    if cf_text == original_text:
        return False
    if abs(len(cf_text) - len(original_text)) < MIN_CF_CHAR_DELTA:
        overlap = len(set(original_text.split()) & set(cf_text.split()))
        denom = max(1, len(set(original_text.split())))
        jacc = overlap / denom
        if jacc > 0.95:
            return False
    return True

print("Prompt templates ready.")

## Section 4 — Fallback API Helpers

In [ ]:
MODEL_USAGE = defaultdict(int)

def _invoke_anthropic(model_id, system, prompt, max_tokens, temperature):
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens"       : max_tokens,
        "temperature"      : temperature,
        "system"           : system,
        "messages"         : [{"role": "user", "content": prompt}],
    })
    resp = bedrock.invoke_model(modelId=model_id, body=body)
    return json.loads(resp["body"].read())["content"][0]["text"].strip()

def _invoke_meta(model_id, prompt, max_tokens):
    llama_prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n{prompt}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
    body = json.dumps({"prompt": llama_prompt, "max_gen_len": max_tokens, "temperature": 0.0})
    resp = bedrock.invoke_model(modelId=model_id, body=body)
    return json.loads(resp["body"].read()).get("generation", "").strip()

def _invoke_mistral(model_id, system, prompt, max_tokens, temperature):
    body = json.dumps({
        "messages": [{"role": "user", "content": f"{system}\n\n{prompt}"}],
        "max_tokens": max_tokens, "temperature": temperature,
    })
    resp = bedrock.invoke_model(modelId=model_id, body=body)
    return json.loads(resp["body"].read())["choices"][0]["message"]["content"].strip()

def _invoke_deepseek(model_id, prompt, max_tokens):
    body = json.dumps({"messages": [{"role": "user", "content": prompt}], "max_new_tokens": max_tokens, "temperature": 0.0})
    resp = bedrock.invoke_model(modelId=model_id, body=body)
    out  = json.loads(resp["body"].read())
    text = out.get("choices", [{}])[0].get("message", {}).get("content", "")
    if "</think>" in text: text = text.split("</think>", 1)[-1]
    return text.strip()

def _invoke_qwen(model_id, prompt, max_tokens):
    body = json.dumps({"messages": [{"role": "user", "content": prompt}], "max_tokens": max_tokens, "temperature": 0.0})
    resp = bedrock.invoke_model(modelId=model_id, body=body)
    return json.loads(resp["body"].read())["choices"][0]["message"]["content"].strip()

def _dispatch(model_name, model_id, family, mode, prompt, max_tokens, temperature):
    if family == "anthropic":
        return _invoke_anthropic(model_id, SYSTEM_GENERATION if mode == "gen" else "", prompt, max_tokens, temperature)
    elif family == "meta":
        return _invoke_meta(model_id, prompt, max_tokens)
    elif family == "mistral":
        return _invoke_mistral(model_id, SYSTEM_GENERATION if mode == "gen" else "", prompt, max_tokens, temperature)
    elif family == "deepseek":
        return _invoke_deepseek(model_id, prompt, max_tokens)
    elif family == "qwen":
        return _invoke_qwen(model_id, prompt, max_tokens)
    else:
        raise ValueError(f"Unknown family: {family}")

def invoke_with_fallback(chain, mode, prompt, max_tokens=1024, temperature=0.7):
    last_error = None
    for model_name, model_id, family in chain:
        for attempt in range(MAX_RETRIES):
            try:
                result = _dispatch(model_name, model_id, family, mode, prompt, max_tokens, temperature)
                MODEL_USAGE[model_name] += 1
                return result, model_name
            except Exception as e:
                last_error = e
                if attempt < MAX_RETRIES - 1:
                    wait = RETRY_DELAY * (attempt + 1)
                    logger.warning(f"{model_name} attempt {attempt+1} failed ({e}). Retry in {wait}s...")
                    time.sleep(wait)
                else:
                    logger.error(f"{model_name} exhausted retries: {e}. Moving to next fallback.")
    raise RuntimeError(f"All fallbacks exhausted. Last error: {last_error}")

def parse_label(text):
    text_upper = text.upper().strip()
    MAPA = {
        "COHERENT": "COHERENT", "COERENTE": "COHERENT",
        "INCOHERENT": "INCOHERENT", "INCOERENTE": "INCOHERENT",
        "PARTIAL": "PARTIAL", "PARCIAL": "PARTIAL",
        "CONTRADICTORY": "CONTRADICTORY", "CONTRADITÓRIO": "CONTRADICTORY", "CONTRADITORIO": "CONTRADICTORY"
    }
    for pt_label, eng_label in MAPA.items():
        if pt_label in text_upper:
            return eng_label
    return "PARSE_ERROR"

print("Fallback API helpers ready.")
print(f"  Generator chain : {[m[0] for m in GENERATOR_CHAIN]}")
print(f"  Validator chain : {[m[0] for m in VALIDATOR_CHAIN]}")

## Section 5 — Checkpoint System (Resume Support)

In [ ]:
processed_records = {}

if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        checkpoint_data = json.load(f)
    processed_records = {r["record_id"]: r for r in checkpoint_data}
    logger.info(f"Checkpoint loaded: {len(processed_records)} records")
    print(f"Checkpoint found : {len(processed_records)} records done | Remaining: {len(anchors) - len(processed_records)}")
else:
    print("No checkpoint found. Starting fresh.")

def save_checkpoint():
    records = list(processed_records.values())
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2, default=str)
    logger.info(f"Checkpoint saved: {len(records)} records")

## Section 6 — Main JACV Generation Loop

> This cell makes API calls via the **fallback chain**. Runtime depends on model latency.
> Every response is tagged with the **actual model** that answered.

In [ ]:
logger.info(f"Starting JACV generation for {len(anchors)} anchors...")

gen_errors = []
val_errors = []
rejected_counterfactuals = []

pbar = tqdm(anchors.itertuples(), total=len(anchors), desc="JACV Generation")

for idx, row in enumerate(pbar):
    record_id = str(row.record_id)

    if record_id in processed_records:
        pbar.set_postfix(status="SKIP (checkpoint)", refresh=False)
        continue

    fato    = str(row.fato)
    direito = str(row.direito)
    pedido  = str(row.pedido)

    jacv_instance = {
        "record_id": record_id,
        "is_recurso": bool(row.is_recurso),
        "fato": fato,
        "direito": direito,
        "pedido_original": pedido,
        "coherence_label_original": "COHERENT",
        "fato_words": int(row.fato_words),
        "direito_words": int(row.direito_words),
        "pedido_words": int(row.pedido_words),
        "counterfactuals": [],
        "jacv_ready": False,
    }

    cf_success = 0

    for ptype in list(PERTURBATION_TYPES.keys()):
        accepted_cf = None

        for regen_round in range(MAX_CF_REGEN + 1):
            cf_entry = {
                "perturbation_type": ptype,
                "intended_label": ptype,
                "pedido_cf": None,
                "generation_model": None,
                "generation_ok": False,
                "validation_model": None,
                "validation_label": None,
                "validation_matches": None,
                "regen_round": regen_round,
            }

            try:
                pbar.set_postfix(record=record_id[-6:], step=f"gen:{ptype[:5]}:{regen_round}", refresh=True)
                prompt_gen = build_generation_prompt(fato, direito, pedido, ptype)
                cf_text, gen_model = invoke_with_fallback(
                    GENERATOR_CHAIN, "gen", prompt_gen, max_tokens=900, temperature=0.75
                )
                cf_entry["pedido_cf"] = cf_text
                cf_entry["generation_model"] = gen_model
                cf_entry["generation_ok"] = True
                time.sleep(API_SLEEP)
            except Exception as e:
                cf_entry["generation_error"] = str(e)
                gen_errors.append({"record_id": record_id, "ptype": ptype, "regen_round": regen_round, "error": str(e)})
                logger.error(f"Gen error {record_id} {ptype} round={regen_round}: {e}")
                continue

            if not is_counterfactual_nontrivial(pedido, cf_text):
                cf_entry["validation_label"] = "REJECTED_TRIVIAL"
                rejected_counterfactuals.append({
                    "record_id": record_id,
                    "ptype": ptype,
                    "reason": "TRIVIAL_OR_COPY",
                    "regen_round": regen_round,
                    "generation_model": gen_model,
                    "pedido_cf": cf_text[:600],
                })
                continue

            try:
                pbar.set_postfix(record=record_id[-6:], step=f"val:{ptype[:5]}:{regen_round}", refresh=True)
                prompt_val = build_validation_prompt(direito, cf_text)
                val_text, val_model = invoke_with_fallback(
                    VALIDATOR_CHAIN, "val", prompt_val, max_tokens=50, temperature=0.0
                )
                val_label = parse_label(val_text)
                cf_entry["validation_model"] = val_model
                cf_entry["validation_label"] = val_label
                if ptype in ["INCOHERENT", "CONTRADICTORY"]:
                    cf_entry["validation_matches"] = (val_label in ["INCOHERENT", "CONTRADICTORY"])
                else:
                    cf_entry["validation_matches"] = (val_label == ptype)
                time.sleep(API_SLEEP)
            except Exception as e:
                cf_entry["validation_error"] = str(e)
                val_errors.append({"record_id": record_id, "ptype": ptype, "regen_round": regen_round, "error": str(e)})
                logger.error(f"Val error {record_id} {ptype} round={regen_round}: {e}")
                continue

            if cf_entry["validation_matches"] is True:
                accepted_cf = cf_entry
                print(f"  [✓ ACCEPTED] {record_id} - {ptype} validado como {val_label} (Round {regen_round})")
                break

            print(f"  [✗ REJEITADO] {record_id} - Gerou {ptype}, mas validador achou {val_label} (Round {regen_round})")
            rejected_counterfactuals.append({
                "record_id": record_id,
                "ptype": ptype,
                "reason": f"VALIDATOR_MISMATCH->{cf_entry['validation_label']}",
                "regen_round": regen_round,
                "generation_model": cf_entry["generation_model"],
                "validation_model": cf_entry["validation_model"],
                "pedido_cf": str(cf_entry["pedido_cf"])[:600],
            })

        if accepted_cf is not None:
            jacv_instance["counterfactuals"].append(accepted_cf)
            cf_success += 1

    jacv_instance["jacv_ready"] = (cf_success == CF_PER_ANCHOR)
    processed_records[record_id] = jacv_instance

    if (idx + 1) % CHECKPOINT_FREQ == 0:
        save_checkpoint()
        pbar.set_postfix(status=f"checkpoint@{idx+1}", refresh=True)

save_checkpoint()

n_ready = sum(1 for r in processed_records.values() if r["jacv_ready"])
n_total = len(processed_records)

print(f"\nGeneration complete.")
print(f"  Records processed         : {n_total}")
print(f"  jacv_ready=True           : {n_ready} ({n_ready/n_total*100:.1f}%)")
print(f"  Generation errors         : {len(gen_errors)}")
print(f"  Validation errors         : {len(val_errors)}")
print(f"  Rejected counterfactuals  : {len(rejected_counterfactuals)}")

for m, c in sorted(MODEL_USAGE.items(), key=lambda x: -x[1]):
    print(f"  {m}: {c} calls")

logger.info(
    f"Generation done: ready={n_ready}/{n_total} | gen_err={len(gen_errors)} | "
    f"val_err={len(val_errors)} | rejected={len(rejected_counterfactuals)}"
)

## Section 7 — Quality Metrics & Validation Audit

In [ ]:
logger.info("Computing quality metrics...")
records = list(processed_records.values())

pt_stats = defaultdict(lambda: {"accepted": 0, "rejected": 0, "total_attempted": 0})
gen_model_usage = Counter()
val_model_usage = Counter()

for rec in records:
    accepted_types = {cf["perturbation_type"] for cf in rec["counterfactuals"]}
    for ptype in PERTURBATION_TYPES.keys():
        if ptype in accepted_types:
            pt_stats[ptype]["accepted"] += 1
        else:
            pt_stats[ptype]["rejected"] += 1
        pt_stats[ptype]["total_attempted"] += 1

    for cf in rec["counterfactuals"]:
        if cf.get("generation_model"):
            gen_model_usage[cf["generation_model"]] += 1
        if cf.get("validation_model"):
            val_model_usage[cf["validation_model"]] += 1

pair_side_counter = Counter()
for rec in records:
    if not rec["jacv_ready"]:
        continue
    for cf in rec["counterfactuals"]:
        swap_pair = bool((hash(f"{rec['record_id']}_{cf['perturbation_type']}_{RANDOM_SEED}") % 2) == 1)
        coherent_side = "B" if swap_pair else "A"
        pair_side_counter[coherent_side] += 1

print("=== Acceptance by Perturbation Type ===")
for ptype, s in pt_stats.items():
    acc_rate = 100 * s["accepted"] / max(1, s["total_attempted"])
    print(f"  {ptype:15s} accepted={s['accepted']:3d} | rejected={s['rejected']:3d} | accept_rate={acc_rate:.1f}%")

print("\n=== Pair Side Balance (expected ~50/50) ===")
for side, count in sorted(pair_side_counter.items()):
    print(f"  coherent_option={side}: {count}")

print("\n=== Model Usage Audit ===")
print("Generator:")
for m, c in gen_model_usage.most_common():
    print(f"  {m}: {c}")
print("Validator:")
for m, c in val_model_usage.most_common():
    print(f"  {m}: {c}")

RESULTS["generation_stats"] = {
    "total_anchors": len(records),
    "jacv_ready": sum(1 for r in records if r["jacv_ready"]),
    "generation_errors": len(gen_errors),
    "validation_errors": len(val_errors),
    "rejected_counterfactuals": len(rejected_counterfactuals),
    "per_type_acceptance": {k: dict(v) for k, v in pt_stats.items()},
    "pair_side_balance": dict(pair_side_counter),
    "gen_model_usage": dict(gen_model_usage),
    "val_model_usage": dict(val_model_usage),
}

### Figure JACV-01a — Generation Success Rate per Perturbation Type

In [ ]:
matplotlib.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300,
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 13, "axes.labelsize": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": "--",
    "figure.facecolor": "white", "axes.facecolor": "#FAFAFA",
})

ptypes = list(pt_stats.keys())
x = np.arange(len(ptypes))

gen_rates = [
    (pt_stats[p]["accepted"] / pt_stats[p]["total_attempted"] * 100)
    if pt_stats[p]["total_attempted"] > 0 else 0
    for p in ptypes
]

colors_gen = ["#2C6E49", "#C77DFF", "#E67E22"][:len(ptypes)]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    x, gen_rates, width=0.5,
    color=colors_gen, alpha=0.88, edgecolor="white"
)

for bar, rate in zip(bars, gen_rates):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{rate:.1f}%",
        ha="center",
        fontsize=11,
        fontweight="bold"
    )

ax.set_xticks(x)
ax.set_xticklabels(ptypes)
ax.set_ylim(0, 115)
ax.set_ylabel("Generation Success Rate (%)")
ax.set_title("Counterfactual Generation Success\nby Perturbation Type", fontweight="bold")

p = FIGURES_DIR / "figJACV01a_generation_success.png"
fig.savefig(p, bbox_inches="tight")
plt.show()
print(f"Saved -> {p}")

### Figure JACV-01b — Cross-Validation Agreement Rate

In [ ]:
w = 0.35

match_rates = [
    (pt_stats[p]["accepted"] / pt_stats[p]["total_attempted"] * 100)
    if pt_stats[p]["total_attempted"] > 0 else 0
    for p in ptypes
]

mism_rates = [
    (pt_stats[p]["rejected"] / pt_stats[p]["total_attempted"] * 100)
    if pt_stats[p]["total_attempted"] > 0 else 0
    for p in ptypes
]

fig, ax = plt.subplots(figsize=(7, 5))

bars_m = ax.bar(
    x - w/2, match_rates, w,
    label="Match (validator agrees)",
    color="#2C6E49", alpha=0.88
)

bars_mm = ax.bar(
    x + w/2, mism_rates, w,
    label="Mismatch",
    color="#C0392B", alpha=0.88
)

for b, r in list(zip(bars_m, match_rates)) + list(zip(bars_mm, mism_rates)):
    if r > 3:
        ax.text(
            b.get_x() + b.get_width()/2,
            b.get_height() + 1,
            f"{r:.0f}%",
            ha="center",
            fontsize=9
        )

ax.set_xticks(x)
ax.set_xticklabels(ptypes)
ax.set_ylim(0, 115)
ax.set_ylabel("Rate (%)")
ax.set_title("Cross-Model Validation Agreement\n(Generator vs. Validator)", fontweight="bold")
ax.legend(fontsize=9)

p = FIGURES_DIR / "figJACV01b_validation_agreement.png"
fig.savefig(p, bbox_inches="tight")
plt.show()
print(f"Saved -> {p}")

### Figure JACV-02 — Model Usage Audit (Which Fallback Was Used)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (usage, title) in zip(axes, [(gen_model_usage, "Generator Calls by Model"),
                                      (val_model_usage, "Validator Calls by Model")]):
    if not usage:
        ax.text(0.5, 0.5, "No data", ha="center", va="center"); ax.set_title(title); continue
    names  = list(usage.keys())
    counts = list(usage.values())
    bars   = ax.barh(names, counts, color="#1B4F72", alpha=0.85)
    ax.invert_yaxis()
    for bar in bars:
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f"{int(bar.get_width())}", va="center", fontsize=9)
    ax.set_xlabel("Number of API Calls")
    ax.set_title(title, fontweight="bold")

fig.suptitle("Figure JACV-02 — Actual Model Usage Audit\n(Which model in the fallback chain answered each call)",
             fontweight="bold")
plt.tight_layout()
p = FIGURES_DIR / "figJACV02_model_usage_audit.png"
fig.savefig(p, bbox_inches="tight"); plt.show()
print(f"Saved -> {p}")

In [ ]:
from pathlib import Path
import pandas as pd
import json

candidate_names = [
    "validated_label", "validator_label", "predicted_label", "final_label",
    "label", "validation_label", "coherence_label", "verdict", "class"
]

root = Path(RESULTS_DIR)

print("Searching in:", root)
print("-" * 80)

def inspect_json_obj(obj, path):
    if isinstance(obj, list) and len(obj) > 0:
        first = obj[0]
        if isinstance(first, dict):
            keys = list(first.keys())
            hits = [k for k in keys if any(c in k.lower() for c in ["label", "pred", "valid", "class", "verdict"])]
            print(f"[JSON-list] {path} | n={len(obj)} | keys={keys[:12]}")
            if hits:
                print("  candidate keys:", hits)
    elif isinstance(obj, dict):
        keys = list(obj.keys())
        hits = [k for k in keys if any(c in k.lower() for c in ["label", "pred", "valid", "class", "verdict"])]
        print(f"[JSON-dict] {path} | keys={keys[:20]}")
        if hits:
            print("  candidate keys:", hits)

for p in root.rglob("*"):
    if not p.is_file():
        continue

    try:
        if p.suffix.lower() == ".csv":
            df_tmp = pd.read_csv(p, nrows=5)
            hits = [c for c in df_tmp.columns if any(cand in c.lower() for cand in ["label", "pred", "valid", "class", "verdict"])]
            print(f"[CSV] {p} | cols={list(df_tmp.columns)}")
            if hits:
                print("  candidate cols:", hits)

        elif p.suffix.lower() == ".json":
            with open(p, "r", encoding="utf-8") as f:
                obj = json.load(f)
            inspect_json_obj(obj, p)

        elif p.suffix.lower() == ".jsonl":
            with open(p, "r", encoding="utf-8") as f:
                line = f.readline().strip()
            if line:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    keys = list(obj.keys())
                    hits = [k for k in keys if any(c in k.lower() for c in ["label", "pred", "valid", "class", "verdict"])]
                    print(f"[JSONL] {p} | keys={keys[:20]}")
                    if hits:
                        print("  candidate keys:", hits)

        elif p.suffix.lower() == ".parquet":
            df_tmp = pd.read_parquet(p)
            hits = [c for c in df_tmp.columns if any(cand in c.lower() for cand in ["label", "pred", "valid", "class", "verdict"])]
            print(f"[PARQUET] {p} | cols={list(df_tmp.columns)}")
            if hits:
                print("  candidate cols:", hits)

    except Exception as e:
        print(f"[SKIP] {p} -> {type(e).__name__}: {e}")

### Figure JACV-03 — Intended vs. Validated Label Heatmap

In [ ]:
from collections import Counter
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ckpt_path = Path(RESULTS_DIR) / "jacv_generation_checkpoint.json"

with open(ckpt_path, "r", encoding="utf-8") as f:
    ckpt = json.load(f)

print(f"Loaded {len(ckpt)} records from {ckpt_path.name}")

def _has_content(v):
    if v is None:
        return False
    if isinstance(v, str):
        return v.strip() != ""
    return bool(v)

def _intended_from_record(rec):
    labels = []
    if _has_content(rec.get("fato")):
        labels.append("FACT")
    if _has_content(rec.get("direito")):
        labels.append("LAW")

    pedido_val = None
    for k in ["pedido", "pedido_original", "pedido_cf"]:
        if k in rec:
            pedido_val = rec.get(k)
            if _has_content(pedido_val):
                break

    if _has_content(pedido_val):
        labels.append("REQUEST")

    return "+".join(labels) if labels else "NONE"

def _validated_from_record(rec):
    for k in [
        "coherence_label_original",
        "validated_label",
        "validator_label",
        "predicted_label",
        "final_label",
        "label",
    ]:
        if k in rec and rec[k] is not None:
            return str(rec[k]).strip()
    return "UNKNOWN"

intended_labels = [_intended_from_record(r) for r in ckpt]
validated_labels = [_validated_from_record(r) for r in ckpt]

print("Sample intended:", intended_labels[:5])
print("Sample validated:", validated_labels[:5])

pairs = list(zip(intended_labels, validated_labels))
pair_counter = Counter(pairs)

all_labels = sorted(set([a for a, _ in pairs] + [b for _, b in pairs]))
confusion_m = pd.DataFrame(0, index=all_labels, columns=all_labels)

for (intended, validated), count in pair_counter.items():
    confusion_m.loc[intended, validated] = count

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    confusion_m,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=ax,
    linewidths=0.5,
    cbar_kws={"label": "Count"}
)

ax.set_xlabel("Validated Label")
ax.set_ylabel("Intended Label")
ax.set_title("Intended vs. Validated Label Heatmap\n(Inter-Model Agreement)", fontweight="bold")

p = FIGURES_DIR / "figJACV03_label_confusion.png"
fig.savefig(p, bbox_inches="tight")
plt.show()
print(f"Saved -> {p}")

## Section 8 — Assemble Final JACV Dataset

In [ ]:
logger.info("Assembling JACV dataset...")
jacv_records = []

for rec in tqdm(records, desc="Assembling"):
    if not rec["jacv_ready"]:
        continue

    record_id   = rec["record_id"]
    fato        = rec["fato"]
    direito     = rec["direito"]
    pedido_orig = rec["pedido_original"]
    is_recurso  = rec["is_recurso"]

    jacv_records.append({
        "jacv_id": f"{record_id}_ORIG",
        "record_id": record_id,
        "task": "JACV-CLS",
        "is_recurso": is_recurso,
        "fato": fato,
        "direito": direito,
        "pedido": pedido_orig,
        "gold_label": "COHERENT",
        "perturbation_type": "NONE",
        "source": "original",
        "validated_by": None,
        "validator_label": None,
        "validator_matches": None,
    })

    valid_cfs = [cf for cf in rec["counterfactuals"] if cf.get("generation_ok") and cf.get("pedido_cf")]

    for cf in valid_cfs:
        jacv_records.append({
            "jacv_id": f"{record_id}_{cf['perturbation_type']}",
            "record_id": record_id,
            "task": "JACV-CLS",
            "is_recurso": is_recurso,
            "fato": fato,
            "direito": direito,
            "pedido": cf["pedido_cf"],
            "gold_label": cf["intended_label"],
            "perturbation_type": cf["perturbation_type"],
            "source": "counterfactual",
            "generation_model": cf.get("generation_model"),
            "validated_by": cf.get("validation_model"),
            "validator_label": cf.get("validation_label"),
            "validator_matches": cf.get("validation_matches"),
        })

        swap_pair = bool((hash(f"{record_id}_{cf['perturbation_type']}_{RANDOM_SEED}") % 2) == 1)

        if swap_pair:
            pedido_A = cf["pedido_cf"]
            pedido_B = pedido_orig
            coherent_option = "B"
        else:
            pedido_A = pedido_orig
            pedido_B = cf["pedido_cf"]
            coherent_option = "A"

        jacv_records.append({
            "jacv_id": f"{record_id}_ADV_{cf['perturbation_type']}",
            "record_id": record_id,
            "task": "JACV-ADV",
            "is_recurso": is_recurso,
            "fato": fato,
            "direito": direito,
            "pedido_A": pedido_A,
            "pedido_B": pedido_B,
            "coherent_option": coherent_option,
            "pair_swapped": swap_pair,
            "perturbation_type_B": cf["perturbation_type"],
            "source": "adversarial_pair",
        })

stats = {
    "total": len(jacv_records),
    "cls_coherent": sum(1 for r in jacv_records if r.get("task") == "JACV-CLS" and r.get("gold_label") == "COHERENT"),
    "cls_incoherent": sum(1 for r in jacv_records if r.get("task") == "JACV-CLS" and r.get("gold_label") == "INCOHERENT"),
    "cls_contradictory": sum(1 for r in jacv_records if r.get("task") == "JACV-CLS" and r.get("gold_label") == "CONTRADICTORY"),
    "adv_pairs": sum(1 for r in jacv_records if r.get("task") == "JACV-ADV"),
    "adv_A_correct": sum(1 for r in jacv_records if r.get("task") == "JACV-ADV" and r.get("coherent_option") == "A"),
    "adv_B_correct": sum(1 for r in jacv_records if r.get("task") == "JACV-ADV" and r.get("coherent_option") == "B"),
}
for k, v in stats.items():
    print(f"  {k:20s}: {v}")

RESULTS["jacv_dataset"] = stats

### Figure JACV-04 — Dataset Composition (Individual)

In [ ]:
flow_rows = [
    ("Raw corpus", len(df)),
    ("Eligible after filters", len(df_eligible)),
    ("Anchor sample", len(anchors)),
    ("Ready anchors", sum(1 for r in records if r.get("jacv_ready", False))),
    ("JACV-CLS instances",
     stats.get("cls_coherent", 0) +
     stats.get("cls_incoherent", 0) +
     stats.get("cls_contradictory", 0)),
    ("JACV-ADV pairs", stats.get("adv_pairs", 0)),
]

flow_df = pd.DataFrame(flow_rows, columns=["stage", "count"])
flow_df.to_csv(FLOW_TABLE_CSV, index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(flow_df["stage"], flow_df["count"], color="#1B4F72", alpha=0.88)
ax.invert_yaxis()

x_offset = max(flow_df["count"]) * 0.02 if len(flow_df) else 1
for bar in bars:
    w = bar.get_width()
    ax.text(w + x_offset, bar.get_y() + bar.get_height()/2, f"{int(w)}", va="center")

ax.set_title("JACV Dataset Flow", fontweight="bold")
ax.set_xlabel("Count")

p = FIGURES_DIR / "figJACV04a_dataset_flow.png"
fig.savefig(p, bbox_inches="tight")
plt.show()
print(f"Saved -> {p}")

source_name = None
source_records = None

if "jacv_records" in globals() and isinstance(jacv_records, list) and len(jacv_records) > 0:
    source_records = jacv_records
    source_name = "jacv_records"
elif "records" in globals() and isinstance(records, list) and len(records) > 0:
    source_records = records
    source_name = "records"

if source_records is None:
    print("Skipping Figure 2: neither `jacv_records` nor `records` has usable data.")
else:
    tasks_df = pd.DataFrame(source_records)
    print(f"Using {source_name}.")
    print("columns:", list(tasks_df.columns))
    print("shape:", tasks_df.shape)

    cls_df = None

    if not tasks_df.empty:
        if "task" in tasks_df.columns:
            cls_df = tasks_df[tasks_df["task"] == "JACV-CLS"].copy()
        elif "gold_label" in tasks_df.columns:
            cls_df = tasks_df[tasks_df["gold_label"].notna()].copy()
        else:
            cls_df = None

    if cls_df is not None and not cls_df.empty and "is_recurso" in cls_df.columns and "gold_label" in cls_df.columns:
        cross = cls_df.groupby(["gold_label", "is_recurso"]).size().unstack(fill_value=0)

        cross = cross.rename(columns={
            False: "First Instance",
            True: "Appeal",
            0: "First Instance",
            1: "Appeal"
        })

        desired_cols = [c for c in ["First Instance", "Appeal"] if c in cross.columns]
        cross = cross[desired_cols]

        fig, ax = plt.subplots(figsize=(7, 5))
        colors = ["#2C6E49", "#C77DFF"][:len(cross.columns)]
        cross.plot(kind="bar", ax=ax, color=colors, alpha=0.88, edgecolor="white")

        ax.set_xlabel("")
        ax.set_ylabel("Count")
        ax.set_title("JACV-CLS Labels by Document Type", fontweight="bold")
        ax.legend(title="Document Type")
        ax.tick_params(axis="x", rotation=15)

        p = FIGURES_DIR / "figJACV04b_cls_by_doctype.png"
        fig.savefig(p, bbox_inches="tight")
        plt.show()
        print(f"Saved -> {p}")
    else:
        print(
            "Skipping Figure 2: could not find usable CLS fields "
            "(`task` or `gold_label`, plus `is_recurso`)."
        )

fig, ax = plt.subplots(figsize=(5, 4))
side_df = pd.Series({
    "A": stats.get("adv_A_correct", 0),
    "B": stats.get("adv_B_correct", 0),
})

bars = ax.bar(side_df.index, side_df.values, color=["#2C6E49", "#C0392B"], alpha=0.88)
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 1,
        f"{int(bar.get_height())}",
        ha="center"
    )

ax.set_title("ADV Coherent Option Balance", fontweight="bold")
ax.set_ylabel("Count")

p = FIGURES_DIR / "figJACV04c_adv_side_balance.png"
fig.savefig(p, bbox_inches="tight")
plt.show()
print(f"Saved -> {p}")

## Section 9 — Export Dataset & Final Summary

In [ ]:
logger.info(f"Saving JACV dataset: {JACV_PATH}")
with open(JACV_PATH, "w", encoding="utf-8") as f:
    json.dump(jacv_records, f, ensure_ascii=False, indent=2, default=str)

file_mb = os.path.getsize(JACV_PATH) / 1e6

dataset_card = {
    "raw_corpus": int(len(df)),
    "eligible_after_filters": int(len(df_eligible)),
    "anchor_sample": int(len(anchors)),
    "ready_anchors": int(sum(1 for r in records if r["jacv_ready"])),
    "jacv_cls_total": int(stats["cls_coherent"] + stats["cls_incoherent"] + stats["cls_contradictory"]),
    "jacv_adv_total": int(stats["adv_pairs"]),
    "adv_A_correct": int(stats["adv_A_correct"]),
    "adv_B_correct": int(stats["adv_B_correct"]),
    "generation_errors": int(len(gen_errors)),
    "validation_errors": int(len(val_errors)),
    "rejected_counterfactuals": int(len(rejected_counterfactuals)),
    "generator_chain": [m[0] for m in GENERATOR_CHAIN],
    "validator_chain": [m[0] for m in VALIDATOR_CHAIN],
}
with open(DATASET_CARD, "w", encoding="utf-8") as f:
    json.dump(dataset_card, f, ensure_ascii=False, indent=2)

audit_pool = []
for rec in records:
    if not rec["jacv_ready"]:
        continue
    for cf in rec["counterfactuals"]:
        audit_pool.append({
            "record_id": rec["record_id"],
            "is_recurso": rec["is_recurso"],
            "perturbation_type": cf["perturbation_type"],
            "fato": rec["fato"][:1200],
            "direito": rec["direito"][:1500],
            "pedido_original": rec["pedido_original"][:1000],
            "pedido_cf": str(cf["pedido_cf"])[:1000],
            "generation_model": cf.get("generation_model"),
            "validation_model": cf.get("validation_model"),
            "validation_label": cf.get("validation_label"),
        })

audit_df = pd.DataFrame(audit_pool)
if len(audit_df) > AUDIT_SAMPLE_SIZE:
    audit_df = (
        audit_df.groupby("perturbation_type", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), AUDIT_SAMPLE_SIZE // len(PERTURBATION_TYPES)), random_state=RANDOM_SEED))
        .reset_index(drop=True)
    )
audit_df.to_csv(AUDIT_SAMPLE_CSV, index=False, encoding='utf-8-sig')

RESULTS["metadata"]["completed_at"] = datetime.now().isoformat()
RESULTS["metadata"]["jacv_file_size_mb"] = float(file_mb)
RESULTS["generation_errors"] = gen_errors
RESULTS["validation_errors"] = val_errors
RESULTS["rejected_counterfactuals"] = rejected_counterfactuals
RESULTS["dataset_card"] = dataset_card

with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, ensure_ascii=False, indent=2, default=str)

sep = "=" * 66
print(sep)
print("  NOTEBOOK 03 - JACV DATASET CONSTRUCTION - SUMMARY")
print(sep)
print(f"  Raw corpus                 : {len(df)}")
print(f"  Eligible after filters     : {len(df_eligible)}")
print(f"  Anchor instances processed : {len(records)}")
print(f"  jacv_ready records         : {sum(1 for r in records if r['jacv_ready'])}")
print(f"  Total JACV instances       : {len(jacv_records)}")
print(f"  Generation errors          : {len(gen_errors)}")
print(f"  Validation errors          : {len(val_errors)}")
print(f"  Rejected counterfactuals   : {len(rejected_counterfactuals)}")
print(f"  JACV dataset saved         : {JACV_PATH}  ({file_mb:.1f} MB)")
print(f"  Results JSON               : {RESULTS_JSON}")
print(f"  Dataset card               : {DATASET_CARD}")
print(f"  Audit sample               : {AUDIT_SAMPLE_CSV}")
print(sep)

logger.info("Notebook 03_JACV_Dataset_Construction COMPLETE")